# Day 3(M1 Day03) 보충 실습 — Router 판정과 관리자 루프·계층 라우팅

**목표**: 그래프의 엣지를 보고 분배 구조가 Router인지 Supervisor인지 판정하고, 결정 계약·관리자 루프·계층 라우팅으로 고친다. 고치기 전과 후를 같은 시험 요청으로 채점해 숫자로 비교한다.

**소재**: 교재 『AI 에이전트 엔지니어링』 8장(공급망 멀티 에이전트)·5장(LangGraph 라우팅) 예제를 분배 구조만 남겨 다시 쓴 축약판 (원본 파일은 학습 배경 표 참고).

**선수**: 본 실습 Part 1(Supervisor)·Part 3(Router). 본 실습(6시간) 밖의 선택 과제다.

## 학습 배경 — 책 예제를 본 실습의 기준으로 읽기

본 실습에서 두 구조를 이렇게 구분했다.

| 판정 기준 | Supervisor | Router |
| --- | --- | --- |
| 결정 시점 | 매 단계마다 | 입구에서 1회 |
| 작업자가 일을 마치면 | 관리자에게 돌아간다 | `END`로 간다 |
| 결정 형식 | `Decision` 계약 (`Literal`) | `RouteDecision` 계약 (`Literal`) |

이번 보충에서 읽는 책 예제 두 개는 다음과 같다. 원본은 한국어판 저장소(https://github.com/TeeDDub/AI-Agent-Engineering)에 있다.

| 문제 | 원본 파일 | 원본이 하는 일 |
| --- | --- | --- |
| 1·2 | `ch08/supply_chain_logistics_multi_agent.py` | `supervisor_node`가 재고·운송·공급업체 전문가 중 한 명을 골라 일을 맡긴다 |
| 3 | `ch05/langgraph_example.py` | 청구(billing)·기술(technical)로 나눈 뒤, 키워드로 하위 처리(환불·성능)를 더 붙인다 |

> 💡 노트북의 코드는 원본을 그대로 옮긴 것이 아니라 **분배 구조만 남겨 다시 쓴 축약판**이다. 원본의 Loki 로깅·Traceloop 추적·도구 15개는 빼고, 전문가마다 모의 도구 2개만 둔다. 모델은 본 실습과 같은 gpt-4o-mini다.

## 오늘의 전체 그림

**오늘의 질문**: 책이 "슈퍼바이저"라고 부른 그래프는 어느 쪽이고, 무엇을 고쳐야 요청을 빠짐없이 처리하나?

```text
[문제 1 판정]   책의 "슈퍼바이저" 그래프 → 엣지로 구조 판정 · 6건 채점 → 결정 계약 + none
      ▼
[문제 2 루프]   전문가가 둘 필요한 요청 → 관리자에게 돌아오는 루프 · 계획은 LLM, 순서는 코드
      ▼
[문제 3 계층]   책의 2단계 라우터 → 한국어 요청에서 하위 분기를 못 타는 원인 → 하위 분류도 계약으로
```

| 문제 | 과제 | 새로 연습하는 것 | 정답본 풀이 |
| --- | --- | --- | --- |
| 1 | 책의 "슈퍼바이저" 판정과 결정 계약 | 그래프 엣지로 구조 판정, 자유 텍스트 결정의 위험, `none` 경로 | 있음 |
| 2 | 관리자 루프로 고치기 | 여러 전문가가 필요한 요청, 판단(LLM)과 순서 계산(코드) 나누기 | 있음 |
| 3 | 계층 라우터의 키워드 분기 | 키워드 분기의 한계, 하위 분류 계약, 기본값이 오분류를 가리는 문제 | 있음 |

> ⚠️ 실습 전 준비: 본 실습과 같은 가상환경·`.env`. 정답본을 처음부터 끝까지 실행하면 LLM을 80번 안팎 부른다 (gpt-4o-mini, 3~5분).

## 0. 환경 준비

본 실습과 같이 판단 역할(관리자·분류기)은 temperature 0으로 둔다. 전문가도 도구를 고르는 일이라 temperature 0을 쓴다.

In [ ]:
import os

# 추적 끄기: 2과목 Day17에서 .env의 LANGSMITH_TRACING을 켰다면 오늘 호출이 전부 전송된다
os.environ["LANGSMITH_TRACING"] = "false"
os.environ["LANGCHAIN_TRACING_V2"] = "false"

from typing import Literal

from dotenv import load_dotenv
from langchain_core.messages import HumanMessage, SystemMessage, ToolMessage
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langgraph.graph import END, START, StateGraph
from pydantic import BaseModel, Field
from typing_extensions import TypedDict

load_dotenv()

llm_strict = ChatOpenAI(model="gpt-4o-mini", temperature=0)  # 판단 역할: 관리자·분류기·전문가

print("준비 완료:", llm_strict.model_name)

### 0-1. 공급망 팀 — 책 8장 예제를 줄인 것 (문제 1·2에서 쓴다)

전문가 3명이 각자 모의 도구 2개를 가진다. 전문가는 요청을 받아 **도구를 직접 호출**하고 결과를 두 문장으로 보고한다. 본 실습의 작업자(프롬프트만 쓰는 노드)와 다른 점이다.

| 전문가 | 도구 |
| --- | --- |
| `inventory` (재고) | `check_inventory`, `place_reorder` |
| `transportation` (운송) | `track_shipment`, `arrange_express` |
| `supplier` (공급업체) | `evaluate_supplier`, `file_quality_claim` |

> 💡 이 셀은 주어진다. `run_specialist(name, request)`는 (보고, 호출한 도구 이름 목록)을 반환한다.

In [ ]:
# 모의 데이터 — 도구가 조회하는 사내 기록
INVENTORY = {"SKU-12345": "현재 재고 50개, 미처리 주문 200개, 공급업체 대성부품"}
SHIPMENTS = {"SH-778": "부산항 출발, 태풍으로 3일 지연"}
SUPPLIERS = {"대성부품": "최근 3개월 불량률 7% (기준 2%), 납기 준수율 81%"}


@tool
def check_inventory(sku: str) -> str:
    """SKU의 현재 재고와 미처리 주문 수를 조회한다."""
    return INVENTORY.get(sku, "기록 없음")


@tool
def place_reorder(sku: str, quantity: int) -> str:
    """SKU를 지정한 수량만큼 재주문한다."""
    return f"{sku} {quantity}개 재주문 접수"


@tool
def track_shipment(shipment_id: str) -> str:
    """배송 번호로 현재 위치와 지연 여부를 조회한다."""
    return SHIPMENTS.get(shipment_id, "기록 없음")


@tool
def arrange_express(shipment_id: str) -> str:
    """지연된 배송을 특급 배송으로 전환한다."""
    return f"{shipment_id} 특급 배송 전환 완료"


@tool
def evaluate_supplier(supplier: str) -> str:
    """공급업체의 불량률과 납기 준수율을 조회한다."""
    return SUPPLIERS.get(supplier, "기록 없음")


@tool
def file_quality_claim(supplier: str, issue: str) -> str:
    """공급업체에 품질 개선 요구(클레임)를 접수한다."""
    return f"{supplier} 품질 클레임 접수: {issue}"


TEAM = {
    "inventory": ("당신은 재고 전문가입니다. 재고를 조회하고 필요하면 재주문합니다.", [check_inventory, place_reorder]),
    "transportation": ("당신은 운송 전문가입니다. 배송을 추적하고 지연되면 특급 배송으로 전환합니다.", [track_shipment, arrange_express]),
    "supplier": ("당신은 공급업체 전문가입니다. 공급업체를 평가하고 품질 문제가 있으면 클레임을 접수합니다.", [evaluate_supplier, file_quality_claim]),
}

# 관리자·분류기가 보는 팀 설명 — 경계가 겹치는 일(발주)은 담당을 적어 둔다
TEAM_GUIDE = """- inventory: 재고 조회와 재주문 (재고를 채우는 발주는 모두 여기)
- transportation: 배송 추적과 특급 배송 전환
- supplier: 공급업체 평가와 품질 클레임 (발주는 하지 않는다)"""


def run_specialist(name, request):
    """전문가가 도구를 호출해 일하고 두 문장으로 보고한다. 반환: (보고, 호출한 도구 이름 목록)"""
    system, tools = TEAM[name]
    by_name = {t.name: t for t in tools}
    messages = [
        SystemMessage(content=system + " 자기 분야의 일만 처리하고 결과를 두 문장으로 보고하세요."),
        HumanMessage(content=request),
    ]
    used = []
    for _ in range(3):  # 도구 호출 왕복은 최대 3번
        ai = llm_strict.bind_tools(tools).invoke(messages)
        messages.append(ai)
        if not ai.tool_calls:
            return ai.content.strip(), used
        for tc in ai.tool_calls:
            result = by_name[tc["name"]].invoke(tc["args"])
            used.append(tc["name"])
            messages.append(ToolMessage(content=str(result), tool_call_id=tc["id"]))
    return "(도구 호출 한도 도달)", used


report, used = run_specialist("transportation", "배송 SH-778이 늦어요. 확인해 주세요.")
print(used)
print(report)

In [ ]:
# 시험 요청 6건과 정답 — 일해야 하는 전문가 (none: 팀 업무가 아니다 → 사람에게 연결)
OPS_TESTS = [
    ("SKU-12345 재고가 몇 개 남았는지 확인하고, 부족하면 재주문해 주세요.", "inventory"),
    ("배송 SH-778이 지연되고 있어요. 상황을 확인하고 필요하면 앞당겨 주세요.", "transportation"),
    ("대성부품 불량이 자꾸 나옵니다. 이 업체를 평가하고 필요하면 조치해 주세요.", "supplier"),
    ("SKU-12345 재고가 바닥났는데 원인이 대성부품 불량이에요. 재주문하고 업체에도 클레임을 걸어 주세요.", "inventory+supplier"),
    ("다음 달 물류팀 회식 장소 좀 추천해 주세요.", "none"),
    ("SH-778 배송 지연 때문에 SKU-12345 재고가 바닥났어요. 배송을 앞당기고 재고도 채워 주세요.", "transportation+inventory"),
]


def expected_workers(label):
    """정답 라벨 → 일해야 하는 전문가 집합 (none이면 빈 집합)"""
    if label == "none":
        return set()
    return set(label.split("+"))


print(len(OPS_TESTS), "건 |", expected_workers("inventory+supplier"))

## 문제 1. 책의 "슈퍼바이저"는 어느 쪽인가 — 판정과 결정 계약

아래 셀은 책 8장의 그래프 구조를 그대로 살린 것이다. `book_supervisor`는 팀원 이름을 **자유 텍스트로** 출력하고, `book_route`는 그 글자가 팀원 이름과 정확히 같을 때만 그 전문가에게 보낸다. 다르면 `END`로 간다.

**할 일**

1. **구조 판정**: `book_graph.get_graph().edges`로 엣지 목록을 출력하고, 이 그래프가 Supervisor인지 Router인지 학습 배경의 기준으로 판정한다. 근거를 한 줄 적는다.
2. **자유 텍스트 결정 시험**: `book_route`에 모델이 낼 법한 출력 4가지(`"inventory"`, `"Inventory."`, `"inventory, supplier"`, `"재고 담당(inventory)"`)를 넣어 어디로 가는지 출력한다. LLM 호출 없이 함수만 부른다.
3. **채점기** `check_workers(result, label)` — 실제로 일한 전문가(`result["worked"]`)가 정답 집합과 같은지 검사해 (맞음 여부, 일한 전문가) 를 반환한다.
4. `OPS_TESTS` 6건을 `book_graph`로 실행해 채점하고, 맞은 수를 출력한다.
5. **결정 계약으로 고치기**: `RouteChoice` 계약(`specialist`: `Literal["inventory", "transportation", "supplier", "none"]`, `reason`)으로 바꾼 라우터와, `none`일 때 가는 `human` 노드(사람에게 연결)를 만든다. 같은 6건으로 다시 채점한다.

**확인 기준**: 고친 그래프에서 팀 밖 요청(5번)이 `human`으로 간다. 전문가가 둘 필요한 요청(4·6번)은 여전히 틀린다 → 그 이유를 한 줄로 적는다 (문제 2로 이어진다).

> 💡 결정 출력이 이번 실행에서 이름만 나왔다고 안전한 것은 아니다. 2번은 모델 출력이 조금 달라졌을 때 그래프가 어떻게 되는지를 함수로 직접 확인하는 시험이다.

In [ ]:
class OpsState(TypedDict):
    request: str
    decision: str  # 관리자의 결정 (책: 자유 텍스트)
    report: str  # 마지막 전문가의 보고
    worked: list[str]  # 일한 전문가 (순서대로)
    log: list[str]


BOOK_PROMPT = f"""당신은 공급망 전문가 팀을 조정하는 슈퍼바이저입니다.
팀원:
{TEAM_GUIDE}

사용자 요청을 기반으로 처리할 팀원 한 명을 선택하세요.
선택한 팀원의 이름(inventory, transportation 또는 supplier)만 출력하고 다른 것은 출력하지 마세요."""


def book_supervisor(state):
    """책의 supervisor_node: 팀원 이름을 자유 텍스트로 출력한다."""
    r = llm_strict.invoke([SystemMessage(content=BOOK_PROMPT), HumanMessage(content=state["request"])])
    return {"decision": r.content, "log": state["log"] + [f"supervisor → {r.content!r}"]}


def book_route(state):
    """책의 route_to_specialist: 이름과 정확히 같을 때만 보내고, 아니면 END"""
    name = state["decision"].strip().lower()
    if name in TEAM:
        return name
    return END


def specialist_node(name):
    """전문가 이름 → 그래프 노드 함수"""

    def node(state):
        report, used = run_specialist(name, state["request"])
        return {
            "report": report,
            "worked": state["worked"] + [name],
            "log": state["log"] + [f"{name}: 도구 {used}"],
        }

    return node


g = StateGraph(OpsState)
g.add_node("supervisor", book_supervisor)
for name in TEAM:
    g.add_node(name, specialist_node(name))
    g.add_edge(name, END)
g.add_edge(START, "supervisor")
g.add_conditional_edges("supervisor", book_route, {"inventory": "inventory", "transportation": "transportation", "supplier": "supplier", END: END})
book_graph = g.compile()

result = book_graph.invoke({"request": OPS_TESTS[0][0], "worked": [], "log": []})
for line in result["log"]:
    print(" ", line)
print("[보고]", result["report"])

**풀이 작성** — 아래 빈 셀에 할 일 순서대로 작성한다. 셀은 필요한 만큼 늘린다.

## 문제 2. 관리자 루프로 고치기 — 여러 전문가가 필요한 요청

4·6번 요청은 전문가 두 명이 차례로 일해야 끝난다. Router는 입구에서 한 명만 고르므로 구조상 처리할 수 없다 → 전문가가 일을 마치면 **관리자에게 돌아오는** 루프로 바꾼다 (본 실습 Part 1의 구조).

**할 일**

1. 계획 계약 `OpsPlan` — 칸: 팀 업무인지(`in_scope`, bool), 일해야 하는 전문가 전부(`needed`, `list[Literal["inventory", "transportation", "supplier"]]`)
2. `LoopState` — 문제 1의 `OpsState` 칸에 `needed`(계획), `planned`(계획을 세웠는지 플래그), `reports`(전문가 보고 목록), `next`(다음 노드)를 더한다.
3. 관리자 노드 `loop_supervisor(state)`
   - 아직 계획이 없으면 LLM으로 `OpsPlan`을 받아 `needed`에 적는다. `in_scope`가 False면 `next="human"`.
   - **다음 전문가는 코드가 계산한다**: `needed` 중 `worked`에 없는 첫 번째. 없으면 `next="done"`.
   - 로그 길이가 `MAX_STEPS`(8) 이상이면 `done`으로 정상 종료한다.
4. 전문가 노드 — 문제 1의 노드에 **앞 전문가의 보고(`reports`)를 요청과 함께** 넘기고, 끝나면 관리자에게 돌아간다.
5. 그래프를 조립해 `OPS_TESTS` 6건을 채점한다 (문제 1의 `check_workers`를 그대로 쓴다).

**확인 기준**: 6건 모두 맞는다. 4번 요청의 로그가 `supervisor → inventory → supervisor → supplier → supervisor → done` 순서로 나온다.

> 💡 다음 전문가까지 LLM에게 고르게 하면, 보고 현황을 보여 줘도 이미 일한 전문가를 다시 부르거나 필요 없는 전문가를 부르는 일이 생긴다 (이 문제를 만들며 시험할 때 실제로 MAX_STEPS까지 반복했다). LLM에게는 **판단**(무엇이 필요한가)만 맡기고, **순서 계산**(아직 안 한 것)은 코드가 한다. Day 2 보충의 "코드가 이미 아는 값은 코드가 채운다"와 같은 원칙이다.

> 💡 관리자가 매 단계 계획을 다시 세우면 계획이 단계마다 바뀔 수 있다 → 계획은 처음 한 번만 세우고 `planned` 플래그로 기록한다 (본 실습 1-1의 `review_done`과 같은 방식).

**풀이 작성** — 아래 빈 셀에 할 일 순서대로 작성한다. 셀은 필요한 만큼 늘린다.

## 문제 3. 계층 라우터의 키워드 분기 — 책 5장 예제

책 5장의 지원 봇은 두 단계로 나눈다.

```text
START → categorize(LLM: billing / technical)
          ├─ billing   → invoice → ("refund"가 메시지에 있으면) refund → END
          └─ technical → login   → ("performance"가 메시지에 있으면) performance → END
```

1단계는 LLM이 자유 텍스트로 분류하고, `billing`·`technical` 어느 쪽 글자도 없으면 **기본값 `technical`** 로 보낸다. 2단계는 영어 키워드가 메시지에 있는지로 정한다. 아래 셀은 이 구조를 그대로 살린 것이다 (각 처리 노드는 지나간 경로만 기록한다).

**할 일**

1. `SUPPORT_TESTS` 6건을 `book_support`로 실행해, 지나간 경로(`path`)가 정답과 같은지 채점한다.
2. 틀린 요청들의 공통점을 보고 원인을 두 줄로 적는다 (2단계 1줄, 1단계 1줄).
3. 고치기
   - 1단계: `TopRoute` 계약 (`group`: `Literal["billing", "technical", "other"]`) — `other`는 `human` 노드로 보낸다.
   - 2단계: 키워드 대신 계약으로 판단한다 — `BillingNeed`(`needs_refund: bool`), `TechNeed`(`needs_performance: bool`)
4. 같은 6건을 2회 실행해 채점하고, 고치기 전과 비교한다.

**확인 기준**: 고친 그래프가 2회 모두 6 / 6이다.

> ⚠️ 원본 예제의 시연 메시지도 한국어("인보이스와 (가능하다면) 환불 관련 도움을 받고 싶습니다")다. 원본을 그대로 실행해도 환불 단계는 지나가지 않는다.

In [ ]:
class SupportState(TypedDict):
    user_message: str
    issue_type: str
    path: list[str]  # 지나간 처리 단계


def book_categorize(state):
    """책의 categorize_issue: 자유 텍스트 분류 + 기본값 technical"""
    prompt = f"이 지원 요청을 'billing' 또는 'technical'로 분류하세요.\n\n메시지: {state['user_message']}"
    kind = llm_strict.invoke(prompt).content.strip().lower()
    if "billing" in kind:
        kind = "billing"
    elif "technical" in kind:
        kind = "technical"
    else:
        kind = "technical"  # 기본값
    return {"issue_type": kind, "path": state["path"] + [f"categorize:{kind}"]}


def step(name):
    """처리 노드 — 지나간 경로만 기록한다 (원본은 모의 결과 문자열을 돌려준다)"""

    def node(state):
        return {"path": state["path"] + [name]}

    return node


def top_router(state):
    return state["issue_type"]


def book_billing_router(state):
    """책의 billing_router: 영어 키워드 검사"""
    if "refund" in state["user_message"].lower():
        return "refund"
    return "end"


def book_tech_router(state):
    """책의 tech_router: 영어 키워드 검사"""
    if "performance" in state["user_message"].lower():
        return "performance"
    return "end"


def build_support(categorize_fn, billing_router, tech_router):
    """분류기·하위 라우터를 바꿔 끼울 수 있는 지원 봇 조립기"""
    s = StateGraph(SupportState)
    s.add_node("categorize", categorize_fn)
    for name in ["invoice", "refund", "login", "performance", "human"]:
        s.add_node(name, step(name))
    s.add_edge(START, "categorize")
    s.add_conditional_edges("categorize", top_router, {"billing": "invoice", "technical": "login", "other": "human"})
    s.add_conditional_edges("invoice", billing_router, {"refund": "refund", "end": END})
    s.add_conditional_edges("login", tech_router, {"performance": "performance", "end": END})
    for name in ["refund", "performance", "human"]:
        s.add_edge(name, END)
    return s.compile()


book_support = build_support(book_categorize, book_billing_router, book_tech_router)

# 시험 요청 6건과 정답 경로 (categorize 다음부터)
SUPPORT_TESTS = [
    ("이번 달 인보이스를 다시 보내 주세요", ["invoice"]),
    ("안녕하세요, 인보이스와 (가능하다면) 환불 관련 도움을 받고 싶습니다.", ["invoice", "refund"]),
    ("비밀번호를 잊어서 로그인이 안 돼요", ["login"]),
    ("로그인은 되는데 대시보드가 너무 느려요", ["login", "performance"]),
    ("I need my invoice and a refund please", ["invoice", "refund"]),
    ("배송지를 바꾸고 싶어요", ["human"]),
]

print(book_support.invoke({"user_message": SUPPORT_TESTS[1][0], "path": []})["path"])

**풀이 작성** — 아래 빈 셀에 할 일 순서대로 작성한다. 셀은 필요한 만큼 늘린다.

## 정리·회고

1. 책 8장의 `supervisor_node` 그래프를 Router로 판정한 근거는? 이름이 아니라 무엇을 봤는가?
2. 문제 1에서 결정 계약으로 고친 뒤에도 4·6번이 틀린 이유는? 문제 2에서 무엇을 바꿔 해결했는가?
3. 문제 3의 키워드 분기와 기본값은 각각 어떤 오분류를 만들었는가? 숫자를 넣는다.

| 오늘 연습한 것 | 다음에 쓰는 곳 |
| --- | --- |
| 그래프 엣지로 구조 판정하기 | 단위 프로젝트에서 다른 사람의 에이전트 코드 읽기 |
| `none`/`other` 경로 — 팀 밖 요청을 사람에게 | M3 가드레일의 사람 승인(Human-in-the-loop) |
| 판단은 LLM, 순서 계산은 코드 | Day 4 병렬 분배에서 결과를 모으는 코드 |
| 같은 시험 요청으로 고치기 전·후 채점 | Day 5 품질 게이트, M2 Day 8 CI 회귀 확인 |